# AIC 2026 — Full embedding ablation on Kaggle

Notebook chạy benchmark **embedding-only** trên 58 KIS chắc chắn:

- OpenCLIP, SigLIP2, Qwen3-VL-Embedding-2B
- full query và perspective n=3/5/7
- Top-100, exact `video_id + frame_id`
- xác nhận OpenCLIP/SigLIP2 có cùng corpus; tự build/resume Qwen image collection nếu thiếu
- chạy từng text encoder trên GPU Kaggle, tuần tự để tránh tràn VRAM
- xuất rank table, Recall@K, MRR và latency

Điều kiện bắt buộc: mỗi model đã có image-vector collection riêng trên Zilliz/Milvus.
Attach Kaggle Dataset `lcdngthnh/aic-2026`; notebook tự tìm cả database và keyframe local trong dataset này.
Không cần Base URL/API key cho embedding vì encoder chạy qua localhost trong notebook.
Không ghi secret trực tiếp vào notebook; dùng **Add-ons → Secrets**.


In [ ]:
# ========================= CONFIG DUY NHẤT =========================
REPO_URL = "https://github.com/zintomvn/Multimodal-Retrieval.git"
BRANCH = "experiment/embedding-ablation-molab"
REPO_DIR = "/kaggle/working/Multimodal-Retrieval"

# Kaggle Dataset đã attach. Để "" để notebook tự tìm trong /kaggle/input.
DATASET_INPUT_DIR = __import__("os").environ.get("ABLATION_DATASET_INPUT_DIR", "")
# lcdngthnh/aic-2026: để trống để tự nhận /kaggle/input/aic-2026 hoặc đường dẫn legacy.
KAGGLE_AIC_INPUT_ROOT = ""
DB_FILENAME = "dev_search_local.db"
GROUNDTRUTH_REL = "data/experiments/aic_2026_groundtruth/aic2026_all_confirmed.csv"
BATCH_MIN = int(__import__("os").environ.get("ABLATION_BATCH_MIN", "0"))
BATCH_MAX = int(__import__("os").environ.get("ABLATION_BATCH_MAX", "0"))
SUBSET_DATASET_ID = __import__("os").environ.get("ABLATION_DATASET_ID", "aic-l01-l25-ablation")

MODELS = ["openclip", "siglip2", "qwen3_vl"]
PERSPECTIVE_COUNTS = [3, 5, 7]
TOP_K = 100
SMOKE_QUERY_COUNT = 6
RUN_FULL_AFTER_SMOKE = True
FRAME_TOLERANCE = 0
MIN_EXPECTED_COLLECTION_ROWS = 300_000
BUILD_QWEN_IF_INCOMPLETE = True
REQUIRE_EXISTING_QWEN_COVERAGE = __import__("os").environ.get("ABLATION_REUSE_QWEN_ONLY", "0") == "1"
RUN_FULL_QWEN_INGEST = True
MAX_AUTO_QWEN_HOURS = 10.0
ALLOW_LONG_QWEN_RUN = True
QWEN_IMAGE_BATCH_SIZE = 8  # mỗi GPU; giảm còn 4 nếu CUDA OOM
QWEN_MAX_PIXELS = 256 * 256

# LLM chỉ dùng một lần để tạo frozen perspectives.
PLANNER_PROFILE = "openai_gpt4o"
PLANNER_API_KEY_SECRET = "OPENAI_API_KEY"
# Muốn dùng Groq: đổi thành groq_gpt_oss_120b và GROQ_API_KEY.

BACKEND_BASE_URL = "http://127.0.0.1:8000"
MODEL_SPECS = {
    "openclip": {
        "model": "ViT-H-14-quickgelu-dfn5b",
        "dim": 1024,
        "collection": "keyframe_embeddings_clip_vith14_quickgelu_dfn5b_v2",
        "base_url": "http://127.0.0.1:8001/v1",
    },
    "siglip2": {
        "model": "ViT-SO400M-16-SigLIP2-384-webli",
        "dim": 1152,
        "collection": "keyframe_embeddings_siglip2_so400m16_384_webli_openclip_1152_v1",
        "base_url": "http://127.0.0.1:8003/v1",
    },
    "qwen3_vl": {
        "model": "Qwen/Qwen3-VL-Embedding-2B",
        "dim": 2048,
        "collection": "keyframe_embeddings_qwen3_vl_embedding_2b_2048_v1",
        "base_url": "http://127.0.0.1:8004/v1",
    },
}

REQUIRED_SECRET_LABELS = ["MILVUS_URI", "MILVUS_TOKEN", PLANNER_API_KEY_SECRET]
# ==================================================================


In [ ]:
import json, os, shutil, sqlite3, subprocess, sys, time
import torch
from pathlib import Path

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, cwd=cwd, env=env, check=True)

print("Python:", sys.version)
run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])

## 1. Clone code và cài dependency

In [ ]:
repo = Path(REPO_DIR)
if not repo.exists():
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo)])
else:
    run(["git", "fetch", "origin", BRANCH], cwd=repo)
    run(["git", "checkout", BRANCH], cwd=repo)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=repo)

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "apps/backend/requirements.txt")])
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "apps/backend/requirements-embedding-service.txt")])
if "qwen3_vl" in MODELS:
    run([
        sys.executable, "-m", "pip", "install", "-q",
        "sentence-transformers>=5.4.0", "transformers>=4.57.3",
        "qwen-vl-utils>=0.0.14", "accelerate>=1.12.0",
    ])


## 2. Nạp Kaggle Secrets (không in giá trị)

In [ ]:
from kaggle_secrets import UserSecretsClient

secret_client = UserSecretsClient()
secret_status = {}
for label in REQUIRED_SECRET_LABELS:
    try:
        value = secret_client.get_secret(label)
    except Exception:
        value = ""
    if value:
        os.environ[label] = value
        secret_status[label] = "configured"
    else:
        secret_status[label] = "missing"

missing = [name for name, status in secret_status.items() if status == "missing"]
if missing:
    raise RuntimeError(f"Thiếu Kaggle Secrets: {missing}")

os.environ["AGENT_LLM_PROFILE"] = PLANNER_PROFILE
os.environ["CLIP_EMBEDDING_BASE_URL"] = MODEL_SPECS["openclip"]["base_url"]
os.environ["SIGLIP2_EMBEDDING_BASE_URL"] = MODEL_SPECS["siglip2"]["base_url"]
os.environ["QWEN3_VL_EMBEDDING_BASE_URL"] = MODEL_SPECS["qwen3_vl"]["base_url"]
os.environ.pop("SIGLIP2_API_KEY", None)
os.environ.pop("QWEN3_VL_EMBEDDING_API_KEY", None)
print(secret_status)


## 3. Tìm database và kiểm tra ground truth

In [ ]:
input_root = Path(DATASET_INPUT_DIR) if DATASET_INPUT_DIR else Path("/kaggle/input")
db_candidates = list(input_root.rglob(DB_FILENAME))
if not db_candidates:
    available_databases = [str(path) for path in input_root.rglob("*.db")]
    raise FileNotFoundError(
        f"Không tìm thấy {DB_FILENAME} trong {input_root}. "
        f"Các file .db tìm được: {available_databases[:20]}"
    )

source_db = db_candidates[0]
target_db = (Path("/kaggle/working") / f"dev_search_l{BATCH_MIN:02d}_l{BATCH_MAX:02d}.db") if BATCH_MAX else (repo / "data/dev_search_local.db")
target_db.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source_db, target_db)

if BATCH_MAX:
    with sqlite3.connect(target_db) as connection:
        connection.row_factory = sqlite3.Row
        source_dataset = connection.execute("SELECT * FROM datasets LIMIT 1").fetchone()
        columns = [row[1] for row in connection.execute("PRAGMA table_info(datasets)")]
        values = dict(source_dataset)
        values["dataset_id"] = SUBSET_DATASET_ID
        if "dataset_code" in values:
            values["dataset_code"] = SUBSET_DATASET_ID
        if "name" in values:
            values["name"] = f"AIC L{BATCH_MIN:02d}-L{BATCH_MAX:02d} ablation"
        placeholders = ",".join("?" for _ in columns)
        connection.execute(
            f"INSERT OR REPLACE INTO datasets ({','.join(columns)}) VALUES ({placeholders})",
            [values.get(column) for column in columns],
        )
        batch_sql = "CAST(SUBSTR(video_id, 2, INSTR(video_id, '_') - 2) AS INTEGER)"
        connection.execute(
            f"UPDATE videos SET dataset_id = ? WHERE {batch_sql} BETWEEN ? AND ?",
            (SUBSET_DATASET_ID, BATCH_MIN, BATCH_MAX),
        )
        selected_videos = connection.execute("SELECT COUNT(*) FROM videos WHERE dataset_id = ?", (SUBSET_DATASET_ID,)).fetchone()[0]
        selected_keyframes = connection.execute(
            f"SELECT COUNT(*) FROM keyframes WHERE is_media_present = 1 AND {batch_sql} BETWEEN ? AND ?",
            (BATCH_MIN, BATCH_MAX),
        ).fetchone()[0]
    print(f"Subset L{BATCH_MIN:02d}-L{BATCH_MAX:02d}: {selected_videos} videos, {selected_keyframes} keyframes")
else:
    selected_keyframes = None

groundtruth = repo / GROUNDTRUTH_REL
assert groundtruth.exists(), f"Thiếu ground truth: {groundtruth}"
print("Database:", source_db, source_db.stat().st_size, "bytes")
print("Ground truth:", groundtruth)

run([
    sys.executable, str(repo / "scripts/run_embedding_ablation.py"),
    "--benchmark-csv", str(groundtruth), "--validate-only",
    "--batch-min", str(BATCH_MIN), "--batch-max", str(BATCH_MAX),
], cwd=repo)

## 4. Chuẩn bị local embedding servers


In [ ]:
def model_server_command(model_key):
    if model_key == "openclip":
        return [
            sys.executable, str(repo / "apps/backend/scripts/serve_openclip_embeddings.py"),
            "--host", "127.0.0.1", "--port", "8001", "--device", "cuda",
            "--model-id", MODEL_SPECS[model_key]["model"],
            "--model", "ViT-H-14-quickgelu", "--pretrained", "dfn5b",
        ]
    if model_key == "siglip2":
        return [
            sys.executable, str(repo / "apps/backend/scripts/serve_openclip_embeddings.py"),
            "--host", "127.0.0.1", "--port", "8003", "--device", "cuda",
            "--model-id", MODEL_SPECS[model_key]["model"],
            "--model", "ViT-SO400M-16-SigLIP2-384", "--pretrained", "webli",
        ]
    if model_key == "qwen3_vl":
        return [
            sys.executable, str(repo / "apps/backend/scripts/serve_qwen3_vl_embeddings.py"),
            "--host", "127.0.0.1", "--port", "8004", "--device", "cuda",
            "--model-id", MODEL_SPECS[model_key]["model"],
        ]
    raise ValueError(model_key)

def start_model_server(model_key):
    log_path = Path(f"/kaggle/working/{model_key}_embedding.log")
    log_handle = open(log_path, "w")
    process = subprocess.Popen(
        model_server_command(model_key), cwd=repo,
        stdout=log_handle, stderr=subprocess.STDOUT, env=os.environ.copy(),
    )
    return process, log_handle, log_path

def stop_model_server(process, log_handle):
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
    log_handle.close()


## 5. Hàm kiểm tra embedding endpoint


In [ ]:
import httpx, math

def wait_embedding_endpoint(model_key, attempts=180):
    spec = MODEL_SPECS[model_key]
    error = None
    for _ in range(attempts):
        try:
            with httpx.Client(timeout=120) as client:
                response = client.post(
                    f"{spec['base_url']}/embeddings",
                    json={"model": spec["model"], "input": ["a person cooking food"]},
                )
                response.raise_for_status()
                vector = response.json()["data"][0]["embedding"]
            if len(vector) != spec["dim"]:
                raise RuntimeError(f"{model_key}: expected dim={spec['dim']}, got {len(vector)}")
            if not vector or not all(math.isfinite(float(x)) for x in vector):
                raise RuntimeError(f"{model_key}: vector rỗng hoặc có NaN/Inf")
            print(model_key, "local endpoint=OK", "dim=", len(vector))
            return
        except Exception as exc:
            error = exc
            time.sleep(2)
    raise RuntimeError(f"Endpoint {model_key} chưa sẵn sàng: {error}")


## 6. Kiểm tra hai baseline collections và trạng thái Qwen


In [ ]:
from pymilvus import MilvusClient

milvus_uri = os.getenv("MILVUS_URI", "")
milvus_token = os.getenv("MILVUS_TOKEN", "")
if not milvus_uri or not milvus_token:
    raise RuntimeError("Thiếu MILVUS_URI hoặc MILVUS_TOKEN trong Kaggle Secrets")

milvus = MilvusClient(uri=milvus_uri, token=milvus_token, timeout=30)

def inspect_collection(model_key, required=True):
    spec = MODEL_SPECS[model_key]
    name = spec["collection"]
    if not milvus.has_collection(collection_name=name):
        if required:
            raise RuntimeError(f"Collection bắt buộc chưa tồn tại: {name}")
        return {"exists": False, "dim": 0, "rows": 0}
    description = milvus.describe_collection(collection_name=name)
    vector_field = next((f for f in description.get("fields", []) if f.get("name") == "vector"), {})
    params = vector_field.get("params") or {}
    dim = int(params.get("dim") or vector_field.get("dim") or 0)
    rows = int(milvus.get_collection_stats(collection_name=name).get("row_count") or 0)
    if dim != spec["dim"]:
        raise RuntimeError(f"{name}: expected dim={spec['dim']}, got {dim}")
    if required and rows < MIN_EXPECTED_COLLECTION_ROWS:
        raise RuntimeError(f"{name}: chỉ có {rows} vectors; cần ít nhất {MIN_EXPECTED_COLLECTION_ROWS}")
    return {"exists": True, "dim": dim, "rows": rows}

def qwen_target_coverage():
    if not BATCH_MAX or not milvus.has_collection(collection_name=MODEL_SPECS["qwen3_vl"]["collection"]):
        return 0
    batch_sql = "CAST(SUBSTR(video_id, 2, INSTR(video_id, '_') - 2) AS INTEGER)"
    with sqlite3.connect(target_db) as connection:
        ids = [row[0] for row in connection.execute(
            f"SELECT keyframe_id FROM keyframes WHERE is_media_present = 1 AND {batch_sql} BETWEEN ? AND ? ORDER BY keyframe_id",
            (BATCH_MIN, BATCH_MAX),
        )]
    found = 0
    collection = MODEL_SPECS["qwen3_vl"]["collection"]
    for start in range(0, len(ids), 1000):
        found += len(milvus.get(collection_name=collection, ids=ids[start:start + 1000], output_fields=["id"]))
    return found

openclip_info = inspect_collection("openclip")
siglip2_info = inspect_collection("siglip2")
if openclip_info["rows"] != siglip2_info["rows"]:
    raise RuntimeError(
        f"Corpus baseline không giống nhau: OpenCLIP={openclip_info['rows']}, "
        f"SigLIP2={siglip2_info['rows']}"
    )
baseline_rows = openclip_info["rows"]
qwen_target_rows = int(selected_keyframes) if selected_keyframes is not None else baseline_rows
qwen_info = inspect_collection("qwen3_vl", required=False)
qwen_covered_rows = qwen_target_coverage() if BATCH_MAX else qwen_info["rows"]

print("OpenCLIP:", openclip_info)
print("SigLIP2:", siglip2_info)
print("Qwen3-VL:", qwen_info)
print("Qwen target vectors:", qwen_target_rows)
print("Qwen covered target vectors:", qwen_covered_rows)
print("Qwen cần build/resume:", qwen_covered_rows < qwen_target_rows)


## 7. Chỉ build/resume Qwen3-VL image embeddings nếu thiếu


In [ ]:
qwen_needs_build = "qwen3_vl" in MODELS and qwen_covered_rows < qwen_target_rows
qwen_checkpoint = Path("/kaggle/working/qwen3_vl_ingest_checkpoint.json")
qwen_ingest_script = repo / "apps/backend/scripts/build_qwen3_vl_image_collection.py"
dataset_root_candidates = [
    Path(KAGGLE_AIC_INPUT_ROOT) if KAGGLE_AIC_INPUT_ROOT else None,
    Path("/kaggle/input/aic-2026"),
    Path("/kaggle/input/datasets/lcdngthnh/aic-2026"),
]
aic_input_root = next((path for path in dataset_root_candidates if path and path.exists()), None)
if qwen_needs_build and aic_input_root is None:
    raise FileNotFoundError("Hãy Add Input dataset lcdngthnh/aic-2026 vào notebook rồi chạy lại.")
print("Local keyframe dataset:", aic_input_root or "not needed — Qwen target coverage is complete")

def run_qwen_ingest(limit=0):
    if not torch.cuda.is_available():
        raise RuntimeError("Kaggle Accelerator phải đặt là GPU.")
    gpu_count = min(2, torch.cuda.device_count())
    launcher = (
        [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={gpu_count}"]
        if gpu_count > 1 else [sys.executable]
    )
    command = launcher + [
        str(qwen_ingest_script),
        "--database", str(target_db),
        "--input-root", str(aic_input_root),
        "--batch-size", str(QWEN_IMAGE_BATCH_SIZE),
        "--max-pixels", str(QWEN_MAX_PIXELS),
        "--checkpoint", str(qwen_checkpoint),
        "--device", "cuda",
    ]
    if BATCH_MAX:
        command.extend(["--batch-min", str(BATCH_MIN), "--batch-max", str(BATCH_MAX), "--allow-baseline-count-mismatch"])
    if limit:
        command.extend(["--limit", str(limit)])
    run(command, cwd=repo, env=os.environ.copy())

if qwen_needs_build:
    if REQUIRE_EXISTING_QWEN_COVERAGE:
        raise RuntimeError(
            f"Qwen L{BATCH_MIN:02d}-L{BATCH_MAX:02d} chưa đủ coverage: "
            f"{qwen_covered_rows}/{qwen_target_rows}. Không sinh thêm vì ABLATION_REUSE_QWEN_ONLY=1."
        )
    if not BUILD_QWEN_IF_INCOMPLETE:
        raise RuntimeError(
            f"Qwen collection chưa đủ: {qwen_info['rows']}/{qwen_target_rows}. "
            "Đặt BUILD_QWEN_IF_INCOMPLETE=True để build/resume."
        )

    # Smoke 10 ảnh, sau đó 1.000 ảnh để đo throughput. Upsert có ID ổn định nên lần sau tự skip.
    run_qwen_ingest(limit=10)
    run_qwen_ingest(limit=1000)
    throughput = json.loads(qwen_checkpoint.read_text(encoding="utf-8"))
    estimated_hours = float(throughput.get("estimated_remaining_hours") or 0.0)
    print(f"Estimated remaining Qwen ingest: {estimated_hours:.2f} hours")

    if RUN_FULL_QWEN_INGEST:
        if estimated_hours > MAX_AUTO_QWEN_HOURS and not ALLOW_LONG_QWEN_RUN:
            raise RuntimeError(
                f"Ước tính {estimated_hours:.2f}h > {MAX_AUTO_QWEN_HOURS:.2f}h. "
                "Kiểm tra throughput rồi đặt ALLOW_LONG_QWEN_RUN=True nếu vẫn muốn tiếp tục."
            )
        run_qwen_ingest()
    else:
        print("RUN_FULL_QWEN_INGEST=False — dừng sau 1.000 ảnh.")

    qwen_info = inspect_collection("qwen3_vl", required=False)
    qwen_covered_rows = qwen_target_coverage() if BATCH_MAX else qwen_info["rows"]

if "qwen3_vl" in MODELS and qwen_covered_rows < qwen_target_rows:
    raise RuntimeError(f"Qwen collection chưa đủ cho target: {qwen_covered_rows} < {qwen_target_rows}")
print("Qwen3-VL collection:", qwen_info)


## 8. Khởi động backend local trong Kaggle

In [ ]:
backend_env = os.environ.copy()
backend_env.update({
    "DATABASE_URL": f"sqlite:///{target_db.resolve()}",
    "MODEL_REGISTRY_PATH": str(repo / "configs/model_registry.yaml"),
    "RETRIEVAL_PROFILES_PATH": str(repo / "configs/retrieval_profiles.yaml"),
    "AGENT_CONFIG_PATH": str(repo / "configs/agent.yaml"),
    "DATA_ROOT": str(repo / "data"),
    "SKIP_DB_INIT": "true",
    "MILVUS_CONNECT_TIMEOUT": "30",
    "MILVUS_SEARCH_TIMEOUT": "120",
})

backend_log_path = Path("/kaggle/working/backend.log")
backend_log = open(backend_log_path, "w")
backend_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=repo / "apps/backend",
    stdout=backend_log,
    stderr=subprocess.STDOUT,
    env=backend_env,
)

for _ in range(60):
    try:
        response = httpx.get(f"{BACKEND_BASE_URL}/healthz", timeout=5)
        if response.status_code == 200:
            print("Backend=OK", response.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError(backend_log_path.read_text(errors="replace")[-5000:])

## 9. Sinh và khóa 7 perspectives cho toàn bộ 58 KIS

In [ ]:
scope_name = f"l{BATCH_MIN:02d}_l{BATCH_MAX:02d}" if BATCH_MAX else "all"
output_root = Path(f"/kaggle/working/embedding_ablation_{scope_name}")
output_root.mkdir(parents=True, exist_ok=True)
perspectives_file = output_root / "perspectives.json"
scope_args = (["--dataset-id", SUBSET_DATASET_ID, "--batch-min", str(BATCH_MIN), "--batch-max", str(BATCH_MAX)] if BATCH_MAX else [])

run([
    sys.executable, str(repo / "scripts/run_embedding_ablation.py"),
    "--benchmark-csv", str(groundtruth),
    "--api-base", BACKEND_BASE_URL,
    *scope_args,
    "--perspective-counts", *map(str, PERSPECTIVE_COUNTS),
    "--perspectives-file", str(perspectives_file),
    "--output-dir", str(output_root / "planning"),
    "--plan-only",
], cwd=repo, env=backend_env)

## 10. Chạy từng model tuần tự: smoke trước, full sau


In [ ]:
smoke_dir = output_root / "smoke_6"
full_dir = output_root / f"full_{scope_name}"

def run_one_benchmark(model_key, destination, limit=0):
    command = [
        sys.executable, str(repo / "scripts/run_embedding_ablation.py"),
        "--benchmark-csv", str(groundtruth),
        "--api-base", BACKEND_BASE_URL,
        *scope_args,
        "--models", model_key,
        "--perspective-counts", *map(str, PERSPECTIVE_COUNTS),
        "--top-k", str(TOP_K),
        "--frame-tolerance", str(FRAME_TOLERANCE),
        "--perspectives-file", str(perspectives_file),
        "--output-dir", str(destination),
        "--timeout-s", "300",
    ]
    if limit:
        command.extend(["--limit", str(limit)])
    run(command, cwd=repo, env=backend_env)

for model_key in MODELS:
    print(f"\n===== {model_key}: loading local encoder =====")
    process, log_handle, log_path = start_model_server(model_key)
    try:
        wait_embedding_endpoint(model_key)
        run_one_benchmark(model_key, smoke_dir / model_key, SMOKE_QUERY_COUNT)
        if RUN_FULL_AFTER_SMOKE:
            run_one_benchmark(model_key, full_dir / model_key)
    except Exception:
        if log_path.exists():
            print(log_path.read_text(errors="replace")[-5000:])
        raise
    finally:
        stop_model_server(process, log_handle)


## 11. Gộp kết quả ba model thành bảng chung


In [ ]:
import importlib.util
import pandas as pd

runner_spec = importlib.util.spec_from_file_location("ablation_runner", repo / "scripts/run_embedding_ablation.py")
runner_module = importlib.util.module_from_spec(runner_spec)
sys.modules[runner_spec.name] = runner_module
runner_spec.loader.exec_module(runner_module)

def combine_results(root, query_limit=0):
    frames = [pd.read_csv(root / model_key / "per_query.csv") for model_key in MODELS]
    combined = pd.concat(frames, ignore_index=True)
    records = combined.to_dict(orient="records")
    queries = runner_module.load_queries(groundtruth)
    queries = runner_module.filter_queries_by_batch(queries, BATCH_MIN, BATCH_MAX)
    if query_limit:
        queries = queries[:query_limit]
    runner_module.write_csv(root / "per_query.csv", records)
    runner_module.write_csv(root / "summary.csv", runner_module.summarize(records))
    runner_module.write_rank_tables(root, queries, records)
    with (root / "raw_responses.jsonl").open("w", encoding="utf-8") as output:
        for model_key in MODELS:
            output.write((root / model_key / "raw_responses.jsonl").read_text(encoding="utf-8"))
    return pd.read_csv(root / "summary.csv")

display(combine_results(smoke_dir, SMOKE_QUERY_COUNT))
print((smoke_dir / "rank_table.md").read_text(encoding="utf-8"))
if RUN_FULL_AFTER_SMOKE:
    display(combine_results(full_dir))
    print((full_dir / "rank_table.md").read_text(encoding="utf-8"))


## 12. Đóng gói kết quả để tải về

In [ ]:
archive = shutil.make_archive(
    f"/kaggle/working/embedding_ablation_results_{scope_name}",
    "zip",
    output_root,
)
print("Download:", archive)
print("Smoke table:", smoke_dir / "rank_table.md")
if RUN_FULL_AFTER_SMOKE:
    print("Full metrics:", full_dir / "summary.csv")
    print("Full LaTeX table:", full_dir / "rank_table.tex")